# Clase 6 — Introducción a Machine Learning con scikit-learn

Hasta ahora trabajamos con modelos de lenguaje que ya vienen entrenados. En esta clase vamos a ver la otra cara del ML: **entrenar nuestro propio modelo desde datos**. No hace falta una GPU ni miles de registros — con un dataset de ejemplo y scikit-learn podés tener un primer modelo funcionando en minutos.

## Contenido

| Sección | Tema |
|---|---|
| 1 | ¿Cuándo tiene sentido usar ML? |
| 2 | Los tres tipos de problemas de ML supervisado |
| 3 | El dataset Iris: nuestro primer problema |
| 4 | Dividir datos: train / test split |
| 5 | Entrenar y evaluar el primer modelo |
| 6 | Qué significan las métricas |
| 7 | Actividad: aplicar el flujo a otro dataset |

---
## 1. ¿Cuándo tiene sentido usar ML?

No todos los problemas necesitan ML. Antes de elegir esta herramienta, vale la pena preguntarse:

| Situación | ¿ML? | Alternativa más simple |
|---|---|---|
| La regla es conocida y no cambia | No | `if/else`, fórmulas |
| Tenés pocos datos (< 100 registros) | Con cuidado | Estadística descriptiva |
| El patrón es complejo y cambia con el tiempo | Sí | — |
| Necesitás predecir a partir de muchas variables | Sí | — |
| Querés detectar patrones que no sabés nombrar | Sí | — |

> 💡 ML es una inversión: requiere datos, tiempo y mantenimiento. Solo tiene sentido cuando el problema no tiene una solución más simple y cuando los datos disponibles realmente contienen la información que necesitás predecir.

---
## 2. Los tres tipos de problemas de ML supervisado

En ML supervisado, el modelo aprende de ejemplos: cada ejemplo tiene **características (features)** y una **etiqueta (label)** que queremos predecir.

| Tipo | La etiqueta es... | Ejemplo |
|---|---|---|
| **Clasificación binaria** | Una de dos categorías | ¿El email es spam o no? |
| **Clasificación multiclase** | Una de varias categorías | ¿De qué especie es esta flor? |
| **Regresión** | Un número continuo | ¿Cuánto va a costar este inmueble? |

In [ ]:
# ─── Importar las librerías que vamos a usar ─────────────────────────────────
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.datasets import load_iris, load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score, classification_report

print("Librerías listas.")

---
## 3. El dataset Iris: nuestro primer problema

El dataset Iris es un clásico de ML. Contiene mediciones de 150 flores de tres especies distintas. La tarea es predecir la especie a partir de las medidas.

| Variable | Qué mide |
|---|---|
| `sepal length` | Largo del sépalo (cm) |
| `sepal width` | Ancho del sépalo (cm) |
| `petal length` | Largo del pétalo (cm) |
| `petal width` | Ancho del pétalo (cm) |
| `species` | Especie (0=Setosa, 1=Versicolor, 2=Virginica) — **etiqueta** |

In [ ]:
# ─── Cargar el dataset ────────────────────────────────────────────────────────
iris = load_iris()

X = iris.data     # matriz de features: (150, 4)
y = iris.target   # vector de etiquetas: 0, 1 o 2

print("Dimensiones del dataset:")
print(f"  X: {X.shape}  → {X.shape[0]} ejemplos, {X.shape[1]} features")
print(f"  y: {y.shape}  → {y.shape[0]} etiquetas")
print()
print("Primeras 5 filas:")
df_iris = pd.DataFrame(X, columns=iris.feature_names)
df_iris["especie"] = [iris.target_names[i] for i in y]  # nombres legibles
df_iris.head()

In [ ]:
# ─── Exploración básica ───────────────────────────────────────────────────────
print("Distribución de especies:")
print(df_iris["especie"].value_counts())
print()
print("Estadísticas descriptivas:")
df_iris.drop(columns="especie").describe().round(2)

In [ ]:
# ─── Visualización rápida ─────────────────────────────────────────────────────
# Ver si las especies son separables visualmente con dos features.

colores = ["steelblue", "coral", "seagreen"]

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

for i, especie in enumerate(iris.target_names):
    mascara = y == i
    ax1.scatter(X[mascara, 0], X[mascara, 1], label=especie, color=colores[i], alpha=0.7)
    ax2.scatter(X[mascara, 2], X[mascara, 3], label=especie, color=colores[i], alpha=0.7)

ax1.set(xlabel="largo sépalo", ylabel="ancho sépalo", title="Sépalo")
ax2.set(xlabel="largo pétalo", ylabel="ancho pétalo", title="Pétalo")

for ax in (ax1, ax2):
    ax.legend()

plt.suptitle("Iris: separabilidad visual por feature pair", fontsize=13)
plt.tight_layout()
plt.show()

# Cuanto más separados están los colores, más fácil es la clasificación

---
## 4. Dividir datos: train / test split

Para evaluar honestamente qué tan bien aprende el modelo, necesitamos datos que él **nunca haya visto durante el entrenamiento**. Si evaluamos en los mismos datos con que entrenamos, el resultado es optimista y engañoso.

La práctica estándar es dividir el dataset en dos partes:

| Conjunto | Proporción típica | Para qué sirve |
|---|---|---|
| **Train** | 70–80% | El modelo aprende de estos ejemplos |
| **Test** | 20–30% | Evaluamos cuánto generalizó lo que aprendió |

In [ ]:
# ─── Train / test split ───────────────────────────────────────────────────────
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,        # 20% para test
    random_state=42,      # fijar semilla para reproducibilidad
    stratify=y            # mantener la misma proporción de clases en ambos conjuntos
)

print("División del dataset:")
print(f"  Train: {X_train.shape[0]} ejemplos")
print(f"  Test:  {X_test.shape[0]} ejemplos")
print()
print("Distribución de clases en train:", dict(zip(*np.unique(y_train, return_counts=True))))
print("Distribución de clases en test: ", dict(zip(*np.unique(y_test,  return_counts=True))))

---
## 5. Entrenar y evaluar el primer modelo

Vamos a usar **KNN (K-Nearest Neighbors)**: el algoritmo más intuitivo para empezar. Para clasificar un punto nuevo, mira los *k* puntos más cercanos en el espacio de features y elige la clase mayoritaria.

En scikit-learn todos los modelos siguen la misma interfaz de 3 pasos: `fit()` → `predict()` → `score()`.

In [ ]:
# ─── Entrenar el modelo ───────────────────────────────────────────────────────
modelo = KNeighborsClassifier(n_neighbors=5)   # k=5: votar entre los 5 vecinos más cercanos

modelo.fit(X_train, y_train)   # aquí ocurre el "aprendizaje" (memorizar los datos de entrenamiento)

print("Modelo entrenado.")

In [ ]:
# ─── Hacer predicciones ───────────────────────────────────────────────────────
y_pred = modelo.predict(X_test)   # predecir sobre datos que el modelo NO vio

print("Predicciones (primeras 10):", y_pred[:10])
print("Etiquetas reales  (primeras 10):", y_test[:10])
print()
print("¿Coinciden?", ["✓" if p == r else "✗" for p, r in zip(y_pred[:10], y_test[:10])])

---
## 6. Qué significan las métricas

In [ ]:
# ─── Accuracy: porcentaje de predicciones correctas ──────────────────────────
acc = accuracy_score(y_test, y_pred)
print(f"Accuracy: {acc:.2%}")
print()

# ─── Classification report: desglose por clase ────────────────────────────────
# Precision: de todos los que predije como clase X, ¿cuántos eran realmente X?
# Recall:    de todos los que eran clase X, ¿cuántos detecté correctamente?
# F1:        promedio armónico de precision y recall (útil cuando las clases están desbalanceadas)

print("Reporte detallado por clase:")
print(classification_report(y_test, y_pred, target_names=iris.target_names))

In [ ]:
# ─── Efecto de k en el accuracy ───────────────────────────────────────────────
# ¿Qué pasa si cambiamos la cantidad de vecinos?

resultados_k = []
for k in range(1, 21):   # probamos k = 1, 2, ..., 20
    m = KNeighborsClassifier(n_neighbors=k)
    m.fit(X_train, y_train)
    acc_k = accuracy_score(y_test, m.predict(X_test))
    resultados_k.append({"k": k, "accuracy": acc_k})

df_k = pd.DataFrame(resultados_k)

plt.figure(figsize=(9, 4))
plt.plot(df_k["k"], df_k["accuracy"], marker="o", color="steelblue")
plt.xlabel("k (número de vecinos)")
plt.ylabel("Accuracy en test")
plt.title("Efecto de k en KNN — dataset Iris")
plt.xticks(range(1, 21))
plt.grid(alpha=0.4)
plt.tight_layout()
plt.show()

mejor_k = df_k.loc[df_k["accuracy"].idxmax()]
print(f"Mejor k: {int(mejor_k['k'])} con accuracy {mejor_k['accuracy']:.2%}")

> 💡 Con k=1 el modelo memoriza perfectamente los datos de entrenamiento pero puede fallar en datos nuevos (sobreajuste). Con k muy alto, promedia demasiado y pierde precisión (subajuste). Hay un punto óptimo en el medio.

---
## 7. Actividad: aplicar el flujo a otro dataset

In [ ]:
# ─── Dataset: breast cancer (diagnóstico médico binario) ──────────────────────
# 569 muestras, 30 features, 2 clases: maligno / benigno

cancer = load_breast_cancer()
X_c = cancer.data
y_c = cancer.target

print("Dataset breast cancer:")
print(f"  {X_c.shape[0]} muestras, {X_c.shape[1]} features")
print(f"  Clases: {cancer.target_names}")
print(f"  Distribución: {dict(zip(*np.unique(y_c, return_counts=True)))}")

In [ ]:
# TODO: Aplicá el mismo flujo que usamos con Iris a este dataset.
# Pasos:
#   1. Dividir X_c, y_c en train y test (80/20, stratify=y_c)
#   2. Entrenar un KNeighborsClassifier con k=5
#   3. Calcular accuracy y classification_report
#   4. Probar al menos 3 valores de k distintos y comparar

# 1. Split
# X_c_train, X_c_test, y_c_train, y_c_test = ...

# 2. Entrenar
# modelo_cancer = ...

# 3. Evaluar
# y_c_pred = ...
# print(accuracy_score(...))
# print(classification_report(...))

# 4. Comparar k
# ...

print("Completá las celdas TODO para continuar.")

---
## Entregable

Guardá el notebook con las celdas ejecutadas.
El entregable es la actividad de la sección 7 completa: flujo aplicado al dataset breast cancer con comparación de al menos 3 valores de k.

**Para la próxima clase:** vamos a convertir estos pasos sueltos en un pipeline reproducible con el objeto `Pipeline` de scikit-learn.